In [1]:
import pandas as pd
from pathlib import Path
from sklearn.ensemble import IsolationForest

processed_path = Path("../data/processed")

feature_data = pd.read_csv(
    processed_path / "daily_features.csv"
)

feature_data["date"] = pd.to_datetime(feature_data["date"])

evaluation_data = feature_data[
    feature_data["day_number"] > 7
].copy()

model_features = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct"
]

# Hold out entire participants, not individual rows
participant_ids = sorted(evaluation_data["Id"].unique())

test_ids = participant_ids[-3:]

train_data = evaluation_data[
    ~evaluation_data["Id"].isin(test_ids)
].copy()

test_data = evaluation_data[
    evaluation_data["Id"].isin(test_ids)
].copy()

print("Train participants:", train_data["Id"].nunique())
print("Test participants:", test_data["Id"].nunique())
print("Train rows:", len(train_data))
print("Test rows:", len(test_data))
print("Test participant IDs:", test_ids)

Train participants: 12
Test participants: 3
Train rows: 65
Test rows: 9
Test participant IDs: [np.int64(7086361926), np.int64(8378563200), np.int64(8792009665)]


In [2]:
X_train = train_data[model_features]
X_test = test_data[model_features]

model = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

model.fit(X_train)

test_data["anomaly_label"] = model.predict(X_test)
test_data["anomaly_score"] = model.decision_function(X_test)

print("Test observations:", len(test_data))
print("\nPredictions:")
print(test_data["anomaly_label"].value_counts())

display(
    test_data[
        [
            "Id",
            "date",
            "steps_deviation_7d_pct",
            "sleep_deviation_7d_pct",
            "anomaly_label",
            "anomaly_score"
        ]
    ].sort_values("anomaly_score")
)

Test observations: 9

Predictions:
anomaly_label
-1    6
 1    3
Name: count, dtype: int64


,Id,date,steps_deviation_7d_pct,sleep_deviation_7d_pct,anomaly_label,anomaly_score
176,7086361926,2016-04-10,-93.127545,-87.755102,-1,-0.214813
178,7086361926,2016-04-12,-94.806405,-3.854875,-1,-0.100385
190,8378563200,2016-04-12,-68.580871,-49.413690,-1,-0.095372
188,8378563200,2016-04-10,-74.533663,19.498227,-1,-0.056152
198,8792009665,2016-04-09,-68.707900,29.037520,-1,-0.051541
187,8378563200,2016-04-09,-78.318284,9.571857,-1,-0.037800
186,8378563200,2016-04-08,-19.177533,-34.333242,1,0.032205
189,8378563200,2016-04-11,-14.463696,-26.506681,1,0.067199
177,7086361926,2016-04-11,-24.608324,16.326531,1,0.079745


In [3]:
test_data["rule_alert"] = (
    (test_data["steps_deviation_7d_pct"] <= -10)
    | (test_data["sleep_deviation_7d_pct"] <= -10)
)

ml_flag = test_data["anomaly_label"] == -1
rule_flag = test_data["rule_alert"]

print("ML anomalies:", ml_flag.sum())
print("Rule alerts:", rule_flag.sum())
print("Both flag:", (ml_flag & rule_flag).sum())
print("ML only:", (ml_flag & ~rule_flag).sum())
print("Rule only:", (~ml_flag & rule_flag).sum())

ML anomalies: 6
Rule alerts: 9
Both flag: 6
ML only: 0
Rule only: 3


## Day 11 — Participant-Level Model Evaluation

### Evaluation Setup
- Training set: 12 participants, 65 observations.
- Test set: 3 unseen participants, 9 observations.
- Isolation Forest was fitted exclusively on the training participants.
- Features: percentage deviation in daily steps and recorded sleep from personal baselines.

### Results
- Isolation Forest flagged 6 of 9 test observations as anomalies.
- The exploratory rule-based method flagged 9 of 9 observations.
- Both methods flagged 6 observations; 3 were flagged only by the rule.
- No observations were flagged exclusively by Isolation Forest.

### Limitations
- The test set is small and contains unequal participant observation counts.
- Ground-truth anomaly or clinical outcome labels are unavailable.
- Recorded sleep may reflect incomplete tracking rather than actual sleep duration.
- These results measure agreement between exploratory methods, not diagnostic accuracy or clinical validity.

### Conclusion
Participant-level holdout evaluation provides an initial check of anomaly detection on unseen participants. Larger datasets, stronger data-quality checks, repeated participant-level validation and independent outcome labels are needed before making performance or health-related claims.